# **II. Analyse critique de l'EPI officiel**
## A. Lien entre richesse et performance environnementale (économétrie)
Partie econometrie : est-ce que l'EPI c'est juste la richesse du pays ?

In [ ]:
import numpy as np
import statsmodels.formula.api as smf
from martin.chargement import charger_donnees, CATEGORIES

df = charger_donnees()
df["log_pib"] = np.log(df["pib_hab"])  # en log sinon le Qatar, le Luxembourg etc écrasent tout

# --- 1) MCO simple EPI ~ log(pib/hab)
# HC1 = ecarts types robustes (l'equivalent du ", robust" sous Stata)
reg1 = smf.ols("EPI ~ log_pib", data=df).fit(cov_type="HC1")
print(reg1.summary())

residu = EPI observé - EPI prédit par la richesse
=> en gros ca mesure l'effort environnemental du pays a richesse donnée (c'est ça qui est intéressant)

In [ ]:
df["residu"] = reg1.resid
colonnes = ["country", "pib_hab", "EPI", "residu"]
print("moins bien que ce que leur richesse prédit :")
print(df.sort_values("residu")[colonnes].head(10))
print("mieux que prévu :")
print(df.sort_values("residu")[colonnes].tail(10))

2) dans l'autre sens : log(pib) sur les 11 categories
pas causal du tout, c'est juste pour voir quel profil ont les pays riches
rq : statsmodels vire tout seul les pays sans score de peche (pays sans littoral) -> N = 127
on avait essayé avec les 40 indicateurs mais trop de variables pour trop peu de pays, ça marchait pas

In [ ]:
reg2 = smf.ols("log_pib ~ " + " + ".join(CATEGORIES), data=df).fit(cov_type="HC1")
print(reg2.summary())

## B. Diversité des profils environnementaux (K-means)
clustering des pays avec k-means sur les 11 categories
idée : voir s'il y a des "profils" de pays et si ça colle avec les continents

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from martin.chargement import charger_donnees, CATEGORIES

df = charger_donnees()

kmeans plante s'il y a des NaN -> on remplace par la moyenne de la colonne
(c'est surtout FSH, la peche, qui manque pour les 44 pays sans acces a la mer. pas parfait mais bon)

In [ ]:
X = df[CATEGORIES].fillna(df[CATEGORIES].mean())
# pas de standardisation, tout est deja entre 0 et 100

# methode du coude pour choisir k
inerties = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    inerties.append(km.inertia_)
plt.plot(range(1, 11), inerties, marker="o")
plt.xlabel("nombre de groupes k")
plt.ylabel("inertie (distance des pays au centre de leur groupe)")
plt.title("Méthode du coude")
plt.show()
# ça se casse vers k=4 donc on part là dessus

df["groupe"] = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(X)

# moyenne de chaque categorie par groupe, c'est ce qui permet de comprendre qui est qui
profils = df.groupby("groupe")[CATEGORIES + ["EPI", "pib_hab"]].mean().round(0)
print(profils.to_string())

# est ce qu'on retrouve les regions ?
print(pd.crosstab(df["region"], df["groupe"]))

for g in range(4):
    print("Groupe", g, ":", ", ".join(df[df["groupe"] == g]["country"]))

# heatmap des profils (lignes = groupes, colonnes = categories)
plt.imshow(profils[CATEGORIES], cmap="Greens")
plt.xticks(range(len(CATEGORIES)), CATEGORIES)
plt.yticks(range(4), ["groupe " + str(g) for g in range(4)])
plt.colorbar(label="score moyen (0-100)", shrink=0.5)
plt.title("Profil moyen de chaque groupe")
plt.show()

carte du monde
contours des pays = natural earth, lu direct en ligne
le /vsizip/vsicurl/ devant c'est parce que sur mon mac python refusait l'url (erreur de certificat SSL)

In [ ]:
monde = gpd.read_file("/vsizip/vsicurl/https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip")
monde = monde[monde["ADM0_A3"] != "ATA"]  # pas besoin de l'antarctique
# merge sur le code pays. attention prendre ADM0_A3 et pas ISO_A3 (France et Norvege y sont a -99 !)
monde = monde.merge("groupe " + df["groupe"].astype(str), left_on="ADM0_A3", right_index=True, how="left")
ax = monde.plot(color="white", edgecolor="lightgrey", figsize=(12, 6))  # en blanc = pas noté dans l'EPI
monde.dropna(subset=["groupe"]).plot(column="groupe", categorical=True, cmap="tab10", legend=True, ax=ax)
ax.set_title("Groupes K-means des pays (11 catégories de l'EPI)")
ax.set_axis_off()
plt.show()

## C. Structure statistique et redondance des catégories (ACP)
ACP sur les 11 categories
les poids de l'EPI sont choisis par des experts -> et si on laissait les données choisir ?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from chargement import charger_donnees, CATEGORIES

df = charger_donnees()
X = df[CATEGORIES].fillna(df[CATEGORIES].mean())  # meme astuce que pour le kmeans (NaN de la peche)

# ACP normée (on centre-réduit, comme en cours) sinon les categories qui varient beaucoup prennent tout
Z = StandardScaler().fit_transform(X)
acp = PCA().fit(Z)

# eboulis des valeurs propres : combien d'axes il faut pour résumer les pays ?
plt.bar(range(1, 12), acp.explained_variance_ratio_ * 100)
plt.xlabel("axe")
plt.ylabel("% de variance expliquée")
plt.title("Eboulis des valeurs propres")
plt.show()
print("variance expliquée par les axes 1 et 2 :", acp.explained_variance_ratio_[:2].round(2))

# poids officiels d'une categorie = son poids x le poids de son objectif (feuille 6)
w = pd.read_excel("raw_data/2022-epi.xlsx", sheet_name="6_EPI_Weights").set_index("Abbreviation")
poids = pd.DataFrame(index=CATEGORIES)
poids["officiel"] = [w.loc[c, "Weight"] * w.loc[w.loc[c, "NextLevel"], "Weight"] for c in CATEGORIES]

# poids "ACP" = coefficients de l'axe 1 ramenés à une somme de 1
# (le signe d'un axe est arbitraire, on le retourne si besoin pour que ce soit positif)
axe1 = acp.components_[0] * np.sign(acp.components_[0].sum())
poids["acp"] = axe1 / axe1.sum()
print(poids.round(3))

# nouvel indice avec les poids de l'ACP, puis on compare les classements
df["epi_acp"] = X @ poids["acp"]
df["rang_officiel"] = df["EPI"].rank(ascending=False)
df["rang_acp"] = df["epi_acp"].rank(ascending=False)
df["places_gagnees"] = df["rang_officiel"] - df["rang_acp"]
print("corrélation des rangs :", round(df["rang_officiel"].corr(df["rang_acp"]), 2))
colonnes = ["country", "EPI", "rang_officiel", "rang_acp", "places_gagnees"]
print(df.sort_values("places_gagnees")[colonnes].head(10))  # ceux qui perdent le plus
print(df.sort_values("places_gagnees")[colonnes].tail(10))  # ceux qui gagnent le plus

# plan factoriel (axes 1 et 2), en couleur les groupes du kmeans (memes reglages que clustering.py)
groupes = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(X)
coord = acp.transform(Z)
plt.scatter(coord[:, 0], coord[:, 1], c=groupes, cmap="tab10")
for pays in ["FRA", "USA", "CHN", "IND", "QAT", "SAU", "TUR", "DNK", "BRA", "NER"]:
    i = df.index.get_loc(pays)
    plt.annotate(pays, (coord[i, 0], coord[i, 1]))
plt.xlabel("axe 1 (" + str(round(acp.explained_variance_ratio_[0] * 100)) + " %)")
plt.ylabel("axe 2 (" + str(round(acp.explained_variance_ratio_[1] * 100)) + " %)")
plt.title("Plan factoriel, couleur = groupe kmeans")
plt.show()

ce qu'on en retient :
- attention l'ACP donne pas un "meilleur" EPI, elle surpondère les categories redondantes (double comptage)
- les 11 categories c'est pas 11 infos independantes : un gros bloc "developpement" (axe 1, 46%)
  + une dimension plutot geographique (axe 2, les pays arides)
- les experts mettent ~23% sur ce bloc et 38% sur le climat -> ca neutralise le double comptage
  et donc le poids de la richesse (le papier le justifie pas comme ça mais c'est ce que ça fait)
- selon la façon de pondérer certains pays bougent de 90 places, le classement dépend beaucoup
  d'un choix que le papier présente comme technique
